# A Hybrid Machine Learning and Deep Learning Approach for Mental Health Detection using Text Analysis

This notebook builds an end-to-end pipeline that flags text which may indicate
depression-related risk, using posts collected from Reddit plus three public
research datasets (a Kaggle Reddit-depression dataset, a Twitter sentiment
dataset, and the SuicideWatch dataset).

**Pipeline overview**

1. Data collection — scraping Reddit's public JSON API
2. Merging in external public datasets and aligning labels
3. Data cleaning & feature engineering
4. Exploratory data analysis (EDA)
5. Text vectorization (Bag-of-Words, TF-IDF)
6. Classical ML models — Logistic Regression, Naive Bayes, Random Forest, SVM,
   KNN, XGBoost, and a Voting Ensemble
7. A rule-assisted inference pipeline — risk scoring, sentiment, keyword
   highlighting, a confidence score, and a plain-language explanation
8. A deep learning baseline (CNN + LSTM) for comparison

> **Disclaimer:** This project is for educational/research purposes only. It is
> **not** a clinical or diagnostic tool, and its predictions should never be
> used as a substitute for professional mental health assessment or support.

## 1. Imports & Configuration

All libraries used anywhere in the notebook are imported once here, along with
the handful of constants (file paths, request headers) that get reused across
sections.

In [ ]:
import os
import re
import string
import time
import pickle
from collections import Counter

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
)
from sklearn.utils import resample

# File paths used throughout the notebook
RAW_DATASET_PATH = "mental_health_dataset.csv"
CLEANED_DATASET_PATH = "mental_health_dataset_cleaned.csv"

# Reddit's public JSON API just wants a descriptive User-Agent
HEADERS = {"User-Agent": "mental-health-research-project"}

## 2. Data Collection — Reddit Scraping

We start with a quick connectivity check against Reddit's public JSON
endpoints, then define **one** reusable scraping function and reuse it for
every collection pass below (a single subreddit, several subreddits,
historical "top" posts, and a full sweep across subreddits × listing modes ×
time filters) instead of copy-pasting the same pagination loop each time.

In [ ]:
# Quick sanity check: can we reach Reddit's JSON API and parse a response?
test_url = "https://www.reddit.com/r/depression/new.json?limit=5"
response = requests.get(test_url, headers=HEADERS)
data = response.json()

for post in data["data"]["children"]:
    print(post["data"]["title"])

In [ ]:
def scrape_subreddit(subreddit, listing="new", time_filter=None, max_pages=10, pause=1):
    """
    Scrape posts from one subreddit's public JSON listing endpoint, handling
    pagination and basic rate-limit back-off.

    Parameters
    ----------
    subreddit   : name of the subreddit, without "r/"
    listing     : "new", "hot", "controversial", or "top"
    time_filter : only used when listing == "top", e.g. "all", "year", "month"
    max_pages   : number of 100-post pages to request
    pause       : seconds to sleep between requests (politeness / rate limits)

    Returns
    -------
    list[dict] — one dict of post fields per post collected
    """
    posts = []
    after = None

    for _ in range(max_pages):
        url = f"https://www.reddit.com/r/{subreddit}/{listing}.json?limit=100"
        if listing == "top" and time_filter:
            url += f"&t={time_filter}"
        if after:
            url += f"&after={after}"

        response = requests.get(url, headers=HEADERS)

        if response.status_code != 200:
            print(f"Rate limited on r/{subreddit} ({listing}) - waiting...")
            time.sleep(5)
            continue

        data = response.json()
        if "data" not in data:
            print(f"No data returned for r/{subreddit} ({listing}), skipping page")
            continue

        children = data["data"]["children"]
        if not children:
            break

        mode_tag = listing if listing != "top" else f"top_{time_filter}"
        for post in children:
            p = post["data"]
            posts.append({
                "post_id": p["id"],
                "title": p["title"],
                "selftext": p["selftext"],
                "created_utc": p["created_utc"],
                "score": p["score"],
                "num_comments": p["num_comments"],
                "subreddit": subreddit,
                "mode": mode_tag,
            })

        after = data["data"]["after"]
        time.sleep(pause)

    return posts


def save_or_merge_dataset(new_df, path=RAW_DATASET_PATH, dedup_col="post_id"):
    """Append new_df onto the dataset on disk (creating it if needed), drop
    duplicates, save, and return the combined DataFrame."""
    if os.path.exists(path):
        old_df = pd.read_csv(path, low_memory=False)
        print(f"Existing dataset size: {len(old_df)}")
        combined = pd.concat([old_df, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=[dedup_col])
    else:
        combined = new_df

    combined.to_csv(path, index=False)
    print(f"Updated dataset size: {len(combined)}")
    return combined


def add_derived_columns(df):
    """Add the text/engagement features shared by every Reddit batch."""
    df = df.copy()
    df["full_text"] = df["title"].fillna("") + " " + df["selftext"].fillna("")
    df["text_length"] = df["full_text"].apply(len)
    df["year"] = pd.to_datetime(df["created_utc"], unit="s").dt.year
    df["engagement_score"] = df["score"] + df["num_comments"]
    return df

In [ ]:
# --- Pass 1: a single subreddit, ~1000 posts ("new" listing, 10 pages) ---
posts = scrape_subreddit("depression", listing="new", max_pages=10)

df = pd.DataFrame(posts)
print(len(df))
df.head()

In [ ]:
df.to_csv("depression_test_1000.csv", index=False)
print("File saved successfully")

In [ ]:
# --- Pass 2: several mental-health-adjacent subreddits, small test batch ---
subreddits = [
    "depression",
    "anxiety",
    "mentalhealth",
    "stress",
    "selfimprovement",
]

all_posts = []
for subreddit in subreddits:
    print(f"Collecting from r/{subreddit}")
    all_posts.extend(scrape_subreddit(subreddit, listing="new", max_pages=3))

df_multi = pd.DataFrame(all_posts)
print(len(df_multi))
df_multi.head()

In [ ]:
# --- Pass 3: historical "top" posts for one subreddit, across time filters ---
time_filters = ["all", "year", "month"]

historical_posts = []
for tf in time_filters:
    print(f"Collecting {tf} data")
    historical_posts.extend(
        scrape_subreddit("depression", listing="top", time_filter=tf, max_pages=5)
    )

df_hist = pd.DataFrame(historical_posts)
print(len(df_hist))
df_hist.head()

In [ ]:
# --- Pass 4: every subreddit x every time filter ("top" posts) ---
batch_posts = []
for subreddit in subreddits:
    for tf in time_filters:
        print(f"Collecting r/{subreddit} | {tf}")
        batch_posts.extend(
            scrape_subreddit(subreddit, listing="top", time_filter=tf, max_pages=5, pause=1)
        )

df_batch = pd.DataFrame(batch_posts)
print(len(df_batch))
df_batch.head()

In [ ]:
combined = save_or_merge_dataset(df_batch)

In [ ]:
# Add the derived text/engagement features onto the dataset built so far
df_full = pd.read_csv(RAW_DATASET_PATH)
df_full = add_derived_columns(df_full)
df_full.to_csv(RAW_DATASET_PATH, index=False)

print(df_full.head())

In [ ]:
# --- Pass 5: a wider sweep — every subreddit x listing mode x time filter ---
modes = ["new", "hot", "controversial"]

big_batch = []
for subreddit in subreddits:
    for mode in modes:
        print(f"Collecting r/{subreddit} | {mode}")
        big_batch.extend(scrape_subreddit(subreddit, listing=mode, max_pages=10, pause=2))

    for tf in time_filters:
        print(f"Collecting r/{subreddit} | top | {tf}")
        big_batch.extend(
            scrape_subreddit(subreddit, listing="top", time_filter=tf, max_pages=10, pause=2)
        )

df_big = pd.DataFrame(big_batch)
print(len(df_big))
df_big.head()

In [ ]:
combined = save_or_merge_dataset(df_big)

In [ ]:
# --- Pass 6: extend coverage to a few more relevant subreddits ---
new_subreddits = [
    "offmychest",
    "TrueOffMyChest",
    "lonely",
    "socialanxiety",
    "healthanxiety",
]

extra_batch = []
for subreddit in new_subreddits:
    for mode in modes:
        print(f"Collecting r/{subreddit} | {mode}")
        extra_batch.extend(scrape_subreddit(subreddit, listing=mode, max_pages=8, pause=2))

    for tf in time_filters:
        print(f"Collecting r/{subreddit} | top | {tf}")
        extra_batch.extend(
            scrape_subreddit(subreddit, listing="top", time_filter=tf, max_pages=8, pause=2)
        )

df_extra = pd.DataFrame(extra_batch)
print("New scraped posts:", len(df_extra))
df_extra.head()

In [ ]:
# Merge the new subreddits in: derive features, align columns, dedupe on full_text
df_main = pd.read_csv(RAW_DATASET_PATH)
df_extra = add_derived_columns(df_extra)
df_extra = df_extra[df_main.columns.intersection(df_extra.columns)]

merged = pd.concat([df_main, df_extra], ignore_index=True)
merged = merged.drop_duplicates(subset=["full_text"])
merged.to_csv(RAW_DATASET_PATH, index=False)

print("Updated dataset size:", len(merged))

In [ ]:
# Sanity check: confirm the file is where we expect it
print(os.getcwd())
os.listdir()

## 3. Merging in External Public Datasets

To boost coverage beyond what we scraped, we fold in three public datasets:

- A Kaggle Reddit depression dataset (`depression_dataset_reddit_cleaned.csv`)
- A Twitter sentiment / depression dataset (`sentiment_tweets3.csv`)
- The SuicideWatch dataset (`Suicide_Detection.csv`)

All three follow the same shape — raw text + a label column under a different
name — so we use one helper to align each of them to our schema
(`full_text`, `label`, `subreddit`, `text_length`, `year`,
`engagement_score`) before merging.

In [ ]:
def align_external_dataset(ext_df, rename_map, label_fn, source_name, main_columns):
    """
    Align an externally-sourced dataset onto the main dataset's schema.

    rename_map  : maps the dataset's own column names -> {"full_text", "label"}
    label_fn    : function applied to the raw label column, returning
                  "depression" or "neutral"
    source_name : tag written into "subreddit" so we can trace provenance
    main_columns: columns of the main dataset, used to keep only shared columns
    """
    ext_df = ext_df.rename(columns=rename_map)
    ext_df["label"] = ext_df["label"].apply(label_fn)
    ext_df["subreddit"] = source_name
    ext_df["text_length"] = ext_df["full_text"].apply(len)
    ext_df["year"] = 0
    ext_df["engagement_score"] = 0
    return ext_df[main_columns.intersection(ext_df.columns)]


def merge_into_main(main_df, new_df, dedup_col="full_text", path=RAW_DATASET_PATH):
    """Concatenate new_df onto main_df, drop duplicate text, and save."""
    merged = pd.concat([main_df, new_df], ignore_index=True)
    merged = merged.drop_duplicates(subset=[dedup_col])
    merged.to_csv(path, index=False)
    return merged

In [ ]:
# --- Kaggle Reddit depression dataset ---
df_main = pd.read_csv(RAW_DATASET_PATH)
kaggle_df = pd.read_csv("depression_dataset_reddit_cleaned.csv")
print("Kaggle dataset size:", len(kaggle_df))

kaggle_df = align_external_dataset(
    kaggle_df,
    rename_map={"clean_text": "full_text", "is_depression": "label"},
    label_fn=lambda x: "depression" if x == 1 else "neutral",
    source_name="kaggle_depression",
    main_columns=df_main.columns,
)

final_df = merge_into_main(df_main, kaggle_df)
print("Final dataset size:", len(final_df))
final_df.head()

In [ ]:
# --- Twitter sentiment / depression dataset ---
df_main = pd.read_csv(RAW_DATASET_PATH)
tweets_df = pd.read_csv("sentiment_tweets3.csv")
print("Tweet dataset size:", len(tweets_df))

tweets_df = align_external_dataset(
    tweets_df,
    rename_map={
        "message to examine": "full_text",
        "label (depression result)": "label",
    },
    label_fn=lambda x: "depression" if x == 1 else "neutral",
    source_name="tweet_dataset",
    main_columns=df_main.columns,
)

final_df = merge_into_main(df_main, tweets_df)
print("FINAL DATASET SIZE:", len(final_df))
final_df.head()

In [ ]:
# --- SuicideWatch dataset ---
df_main = pd.read_csv(RAW_DATASET_PATH)
sw_df = pd.read_csv("Suicide_Detection.csv")
print("SuicideWatch dataset size:", len(sw_df))

sw_df = align_external_dataset(
    sw_df,
    rename_map={"text": "full_text", "class": "label"},
    label_fn=lambda x: "depression" if x == "suicide" else "neutral",
    source_name="suicidewatch_dataset",
    main_columns=df_main.columns,
)

final_big_df = merge_into_main(df_main, sw_df)
print("FINAL MASSIVE DATASET SIZE:", len(final_big_df))

In [ ]:
df_check = pd.read_csv(RAW_DATASET_PATH, low_memory=False)
print(df_check["label"].value_counts())
print(df_check.columns)

## 4. Label Repair

A handful of rows ended up with a missing label after the merges above (some
columns don't line up perfectly across sources). We patch this in two passes:

1. Re-derive the label for SuicideWatch-sourced rows by re-joining on
   `full_text`.
2. For any subreddit-sourced row still missing a label, infer it from which
   subreddit it came from, then default anything left over to `"neutral"`.

In [ ]:
df_final = pd.read_csv(RAW_DATASET_PATH, low_memory=False)

sw_original = pd.read_csv("Suicide_Detection.csv")
sw_original = sw_original.rename(columns={"text": "full_text", "class": "label"})
sw_original["label"] = sw_original["label"].apply(
    lambda x: "depression" if x == "suicide" else "neutral"
)
sw_labels = sw_original[["full_text", "label"]].rename(columns={"label": "label_from_sw"})

df_fixed = df_final.merge(sw_labels, on="full_text", how="left")
df_fixed["label"] = df_fixed["label"].fillna(df_fixed["label_from_sw"])
df_fixed["label"] = df_fixed["label"].fillna("unknown")
df_fixed = df_fixed.drop(columns=["label_from_sw"])

df_fixed.to_csv(RAW_DATASET_PATH, index=False)

print("Label column restored.")
print(df_fixed["label"].value_counts())

In [ ]:
df = pd.read_csv(RAW_DATASET_PATH, low_memory=False)

# Map any still-"unknown" label using which subreddit the post came from
label_map = {
    "depression": "depression",
    "anxiety": "depression",
    "stress": "depression",
    "mentalhealth": "depression",
    "offmychest": "neutral",
    "TrueOffMyChest": "neutral",
    "lonely": "depression",
    "socialanxiety": "depression",
    "healthanxiety": "depression",
    "selfimprovement": "neutral",
}

df.loc[df["label"] == "unknown", "label"] = df["subreddit"].map(label_map)
df["label"] = df["label"].fillna("neutral")

df.to_csv(RAW_DATASET_PATH, index=False)
print(df["label"].value_counts())

## 5. Dataset Summary Report

A quick text summary of the merged dataset, written to
`dataset_summary.txt` for the record.

In [ ]:
df = pd.read_csv(RAW_DATASET_PATH, low_memory=False)

total_rows = len(df)
columns = list(df.columns)
label_counts = df["label"].value_counts()
subreddit_counts = (
    df["subreddit"].value_counts() if "subreddit" in df.columns
    else "Subreddit column not available."
)

summary_text = f"""
MENTAL HEALTH DATASET SUMMARY
=============================

Total Samples: {total_rows}

Columns:
{columns}

Label Distribution:
{label_counts}

Subreddit Distribution:
{subreddit_counts}
"""

with open("dataset_summary.txt", "w") as f:
    f.write(summary_text)

print("dataset_summary.txt created successfully.")

## 6. Data Cleaning

Starting fresh from the saved CSV (so this section can be re-run on its own),
we drop empty rows, normalize the text, and filter out anything too short to
carry signal.

In [ ]:
df = pd.read_csv(RAW_DATASET_PATH, low_memory=False)
print("Dataset shape:", df.shape)
df.head()

In [ ]:
df = df.dropna(subset=["full_text"])
df = df[df["full_text"].str.strip() != ""]

print("After removing empty rows:", df.shape)

In [ ]:
def clean_text(text):
    """Lowercase, strip URLs/punctuation/digits, and collapse whitespace."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df["clean_text"] = df["full_text"].apply(clean_text)
df.head()

In [ ]:
df = df[df["clean_text"].str.len() > 20]
print("After removing short texts:", df.shape)

In [ ]:
df.to_csv(CLEANED_DATASET_PATH, index=False)
print("Cleaned dataset saved successfully.")

## 7. Exploratory Data Analysis

Reloading the cleaned dataset as a checkpoint, then looking at label balance,
where the data came from, text length, and the most common words per class.

In [ ]:
df = pd.read_csv(CLEANED_DATASET_PATH)
print("Dataset shape:", df.shape)
df.head()

In [ ]:
print(df["label"].value_counts())

In [ ]:
plt.figure(figsize=(7, 5))
df["label"].value_counts().plot(kind="bar")

plt.title("Label Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.tight_layout()

plt.savefig("label.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
df["subreddit"].value_counts().head(10)

In [ ]:
df["subreddit"].value_counts().head(10).plot(kind="bar")

plt.title("Top Subreddits in Dataset")
plt.xlabel("Subreddit")
plt.ylabel("Post Count")
plt.tight_layout()
plt.show()

In [ ]:
df["clean_text"].str.len().describe()

In [ ]:
df["clean_text"].str.len().hist(bins=50)

plt.title("Text Length Distribution")
plt.xlabel("Text Length")
plt.ylabel("Frequency")
plt.show()

In [ ]:
all_words = " ".join(df["clean_text"]).split()
word_counts = Counter(all_words)
word_counts.most_common(20)

In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud

text_blob = " ".join(df["clean_text"])
wordcloud = WordCloud(width=800, height=400).generate(text_blob)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Word Cloud of Dataset")
plt.show()

In [ ]:
depression_text = " ".join(df[df["label"] == "depression"]["clean_text"])
neutral_text = " ".join(df[df["label"] == "neutral"]["clean_text"])

print("Depression sample text preview:", depression_text[:200])

## 8. Feature Engineering & Vectorization

Checkpoint-reload the cleaned dataset, then build the features used for
modeling:

- A quick **Bag-of-Words** pass, just to sanity-check vocabulary size
  (not used for the final models below)
- A small set of strongly-worded neutral examples, added to balance out
  some of the noisier "neutral" labels
- The **TF-IDF** matrix that the classical models are actually trained on
- A saved `LabelEncoder` for the `depression` / `neutral` labels

In [ ]:
df = pd.read_csv(CLEANED_DATASET_PATH)
print(df.shape)
df.head()

In [ ]:
y = df["label"]
print(y.value_counts())

In [ ]:
# Exploratory only — shows the Bag-of-Words vocabulary size; not used downstream
bow_vectorizer = CountVectorizer(max_features=5000, stop_words="english")
X_bow = bow_vectorizer.fit_transform(df["clean_text"])

print("BoW feature matrix shape:", X_bow.shape)

In [ ]:
# A few unambiguous positive examples to help the model anchor "neutral"
extra_neutral = [
    "I am happy",
    "I love my life",
    "Today is a great day",
    "I feel excited and motivated",
    "Everything is going well",
    "I am feeling positive",
    "Life is beautiful",
    "I am enjoying my day",
    "I feel great today",
    "I am very satisfied with my work",
]

extra_df = pd.DataFrame({
    "clean_text": extra_neutral,
    "label": ["neutral"] * len(extra_neutral),
})

df = pd.concat([df, extra_df], ignore_index=True)
print("Added neutral samples")

In [ ]:
# Final feature set used for modeling: TF-IDF with unigrams + bigrams
vectorizer = TfidfVectorizer(
    max_features=10000,
    stop_words="english",
    ngram_range=(1, 2),  # include short phrases, not just single words
    min_df=2,            # drop terms that only appear once (likely noise)
)

X_tfidf = vectorizer.fit_transform(df["clean_text"])
y = df["label"]

print("TF-IDF feature matrix shape:", X_tfidf.shape)

In [ ]:
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

with open("encoder.pkl", "wb") as f:
    pickle.dump(encoder, f)

print("Label mapping:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))
print("Encoder saved")

In [ ]:
# Explored class balancing via downsampling the majority class.
# (Note: the models trained below use the full, unbalanced TF-IDF matrix —
# this cell is kept as a documented experiment, not part of the final pipeline.)
df_dep = df[df["label"] == "depression"]
df_neu = df[df["label"] == "neutral"]

df_dep_down = resample(df_dep, replace=False, n_samples=len(df_neu), random_state=42)

df_balanced = pd.concat([df_dep_down, df_neu])
df_balanced = df_balanced.sample(frac=1, random_state=42)

print(df_balanced["label"].value_counts())

## 9. Model Training

We split the TF-IDF matrix into train/test sets (keeping the original
DataFrame index alongside the split so we can trace misclassified examples
back to their text later), then train a spread of classical ML models:
Logistic Regression, Naive Bayes, Random Forest, SVM, KNN, and a Voting
Ensemble — plus XGBoost as an extra comparison point.

In [ ]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_tfidf, y_encoded, df.index, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape)
print("Testing samples:", X_test.shape)

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
print("Logistic Regression trained successfully.")

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
print("Naive Bayes trained successfully.")

rf_model = RandomForestClassifier(n_estimators=100)
rf_model.fit(X_train, y_train)
print("Random Forest trained successfully.")

svm_model = LinearSVC()
svm_model.fit(X_train, y_train)
print("SVM model trained successfully.")

## 10. Model Evaluation

Accuracy, a classification report, and a confusion matrix for each model,
followed by a side-by-side comparison chart, cross-validation, the words
that most influence the Logistic Regression model, and a look at a handful
of misclassified examples (traced back to the original text via the
`idx_test` indices saved during the split).

In [ ]:
def evaluate_model(name, y_true, y_pred):
    print(f"{name} Accuracy:", accuracy_score(y_true, y_pred))
    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred))
    print("\nConfusion Matrix:\n")
    print(confusion_matrix(y_true, y_pred))
    print("\n" + "-" * 60 + "\n")


y_pred_log = log_model.predict(X_test)
y_pred_nb = nb_model.predict(X_test)
y_pred_svm = svm_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

evaluate_model("Logistic Regression", y_test, y_pred_log)
evaluate_model("Naive Bayes", y_test, y_pred_nb)
evaluate_model("SVM", y_test, y_pred_svm)
evaluate_model("Random Forest", y_test, y_pred_rf)

In [ ]:
models = ["Logistic Regression", "Naive Bayes", "SVM", "Random Forest"]
accuracies = [
    accuracy_score(y_test, y_pred_log),
    accuracy_score(y_test, y_pred_nb),
    accuracy_score(y_test, y_pred_svm),
    accuracy_score(y_test, y_pred_rf),
]

print("Model Performance Comparison")
for name, acc in zip(models, accuracies):
    print(f"{name}: {acc}")

plt.figure(figsize=(8, 5))
plt.bar(models, accuracies)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.xlabel("Models")
plt.tight_layout()
plt.savefig("model_accuracy_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
scores = cross_val_score(log_model, X_tfidf, y_encoded, cv=5)
print("Cross Validation Scores:", scores)
print("Average CV Score:", scores.mean())

In [ ]:
feature_names = vectorizer.get_feature_names_out()
coefficients = log_model.coef_[0]

top_positive_indices = np.argsort(coefficients)[-20:]
top_negative_indices = np.argsort(coefficients)[:20]

print("Top words indicating DEPRESSION:\n")
for i in top_positive_indices:
    print(feature_names[i])

print("\nTop words indicating NEUTRAL:\n")
for i in top_negative_indices:
    print(feature_names[i])

In [ ]:
# Trace misclassified test examples back to their original text via idx_test
errors_mask = y_test != y_pred_log
misclassified_idx = idx_test[errors_mask]

print("Number of misclassified samples:", errors_mask.sum())

wrong_true = y_test[errors_mask]
wrong_pred = y_pred_log[errors_mask]

for i, idx in enumerate(misclassified_idx[:10]):
    actual_label = encoder.inverse_transform([wrong_true[i]])[0]
    predicted_label = encoder.inverse_transform([wrong_pred[i]])[0]

    print("\nTEXT:", df.loc[idx, "clean_text"])
    print("Actual:", actual_label)
    print("Predicted:", predicted_label)

### 10.1 Additional Models — KNN, ROC Curve, XGBoost, Voting Ensemble

A few extra models and diagnostics, evaluated on the same train/test split.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)
print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))

In [ ]:
y_prob = log_model.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.savefig("roc_curve.png", dpi=300)
plt.show()

In [ ]:
!pip install xgboost

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(use_label_encoder=False, eval_metric="logloss")
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))

In [ ]:
ensemble = VotingClassifier(
    estimators=[
        ("lr", log_model),
        ("nb", nb_model),
        ("svm", svm_model),
        ("rf", rf_model),
    ],
    voting="hard",
)

ensemble.fit(X_train, y_train)
y_pred_ens = ensemble.predict(X_test)

print("Ensemble Accuracy:", accuracy_score(y_test, y_pred_ens))

## 11. Save the Final Model

Logistic Regression gives a strong, well-calibrated baseline with the
benefit of `predict_proba` (used for the ROC curve and the confidence score
later), so it's the model we ship. We save it alongside the fitted TF-IDF
vectorizer so both can be reloaded for inference without retraining.

In [ ]:
with open("model.pkl", "wb") as f:
    pickle.dump(log_model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Saved successfully")

## 12. Inference Pipeline

This section simulates how the saved model would be used in production: load
the pickled encoder/model/vectorizer, then run new text through cleaning,
vectorization, and prediction. A simple keyword override sends obviously
positive text straight to "Neutral" before it ever reaches the model, since
those phrases are common false positives for a model trained mostly on
distressed text.

In [ ]:
with open("encoder.pkl", "rb") as f:
    encoder = pickle.load(f)

with open("model.pkl", "rb") as f:
    model = pickle.load(f)

with open("vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)


POSITIVE_WORDS = {
    "happy", "great", "excited", "good", "love", "enjoy",
    "joyful", "delighted", "content", "satisfied", "optimistic",
    "cheerful", "glad", "positive", "fantastic", "wonderful",
    "amazing", "pleasant", "peaceful", "motivated",
}


def predict_with_score(text):
    """Classify text as Depression/Neutral, with a decision-function score
    and a coarse risk level. Obvious positive language short-circuits to a
    confident "Neutral" / "Low Risk" result."""
    cleaned = clean_text(text)
    words = set(cleaned.split())

    if words & POSITIVE_WORDS:
        return "Neutral", 1.0, "Low Risk"

    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    score = model.decision_function(vec)[0]
    label = encoder.inverse_transform([pred])[0]

    if score < -1:
        risk = "High Risk"
    elif score < 0:
        risk = "Moderate Risk"
    else:
        risk = "Low Risk"

    return label.capitalize(), score, risk


# Quick check
print(predict_with_score("I feel empty and alone"))
print(predict_with_score("I am happy"))

## 13. Sentiment, Keyword Highlighting, Confidence & Explanations

A few lightweight, rule-based add-ons layered on top of the model's
prediction, all combined in the "final system" cell at the end of this
section:

- **Sentiment** via TextBlob's polarity score
- **Keyword highlighting** — which depression/positive words appear in the text
- **Confidence** — the model's decision-function score squeezed into a 0–100% range via a sigmoid
- **Explanation** — a one-line, plain-language summary of the result

In [ ]:
from textblob import TextBlob


def get_sentiment(text):
    polarity = TextBlob(text).sentiment.polarity
    if polarity > 0:
        return "Positive"
    elif polarity < 0:
        return "Negative"
    return "Neutral"


print(get_sentiment("I am very happy"))
print(get_sentiment("I feel very sad"))

In [ ]:
DEPRESSION_WORDS = {
    "sad", "tired", "empty", "hopeless", "lonely",
    "worthless", "depressed", "cry", "pain", "lost",
    "alone", "fear", "hurt", "broken",
}


def highlight_keywords(text):
    """Return (depression_keywords_found, positive_keywords_found) in text."""
    words = set(clean_text(text).split())
    dep_found = [w for w in words if w in DEPRESSION_WORDS]
    pos_found = [w for w in words if w in POSITIVE_WORDS]
    return dep_found, pos_found

In [ ]:
def get_confidence(score):
    """Squash the decision-function score into a 0-100% confidence value."""
    prob = 1 / (1 + np.exp(-abs(score)))
    return round(prob * 100, 2)

In [ ]:
def generate_explanation(label, sentiment, dep_keywords, pos_keywords):
    """A short, plain-language summary of why the prediction came out this way."""
    if label == "Depression":
        return "The text shows signs of emotional distress and negative expressions."
    elif sentiment == "Positive":
        return "The text reflects a positive emotional state."
    return "The text appears neutral with no strong emotional indicators."

In [ ]:
def analyze_paragraph(text):
    """Run sentence-level risk + sentiment analysis across a paragraph."""
    sentences = text.split(".")
    results = []

    for s in sentences:
        s = s.strip()
        if s:
            label, score, risk = predict_with_score(s)
            sentiment = get_sentiment(s)
            results.append((s, label, sentiment, risk))

    return results

## 14. Final Combined System

The interactive demo: every function above wired together into a single
loop. Type some text and get a prediction, sentiment, risk level, the
keywords that drove the decision, a confidence score, and a plain-language
explanation. Type `exit` to stop.

In [ ]:
while True:
    text = input("\nEnter text (or type 'exit'): ")

    if text.lower() == "exit":
        break

    label, score, risk = predict_with_score(text)
    sentiment = get_sentiment(text)
    dep_keywords, pos_keywords = highlight_keywords(text)
    confidence = get_confidence(score)
    explanation = generate_explanation(label, sentiment, dep_keywords, pos_keywords)

    print("\nPrediction:", label)
    print("Sentiment:", sentiment)
    print("Risk Level:", risk)
    print("Depression Indicators:", dep_keywords)
    print("Positive Indicators:", pos_keywords)
    print("Confidence:", f"{confidence}%")
    print("Explanation:", explanation)

## 15. Deep Learning Baseline — CNN + LSTM

As a comparison point against the classical models above, we train a small
CNN + LSTM network on tokenized, padded sequences instead of TF-IDF
features.

In [ ]:
!pip install tensorflow

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

texts = df["clean_text"].astype(str)
y_dl = df["label"].map({"depression": 0, "neutral": 1})

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
X_pad = pad_sequences(sequences, maxlen=100)

print("Data prepared")

In [ ]:
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(
    X_pad, y_dl, test_size=0.2, random_state=42
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense

model_dl = Sequential()
model_dl.add(Embedding(input_dim=10000, output_dim=128, input_length=100))
model_dl.add(Conv1D(64, 5, activation="relu"))
model_dl.add(MaxPooling1D(pool_size=2))
model_dl.add(LSTM(64))
model_dl.add(Dense(1, activation="sigmoid"))

model_dl.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)

model_dl.summary()

In [ ]:
history = model_dl.fit(
    X_train_dl,
    y_train_dl,
    epochs=3,
    batch_size=64,
    validation_data=(X_test_dl, y_test_dl),
)

In [ ]:
loss, acc = model_dl.evaluate(X_test_dl, y_test_dl)
print("CNN + LSTM Accuracy:", acc)

In [ ]:
train_acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
epochs_range = range(1, len(train_acc) + 1)

plt.figure()
plt.plot(epochs_range, train_acc, label="Training Accuracy")
plt.plot(epochs_range, val_acc, label="Validation Accuracy")

plt.title("CNN + LSTM Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.savefig("cnn_lstm_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()